# 🧠 Átomo LLM — Fine-tuning

Entrena tu propio modelo de lenguaje para Átomo Studio.

**Basado en:** Llama 3.2 3B + Unsloth
**Dataset:** `atomo-llm-dataset.jsonl`
**Duración:** ~1-2 horas
**GPU:** T4 (gratis en Colab)

---
### 📋 Instrucciones
1. Ejecuta esta celda para instalar dependencias
2. Sube tu archivo `atomo-llm-dataset.jsonl` cuando te lo pida
3. Ejecuta todo en orden
4. Al final, descarga tu modelo fine-tuneado

In [ ]:
# ⚡ 1. Instalar Unsloth + dependencias
%%capture
import torch
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("✅ Instalación completa")

In [ ]:
# 📤 2. Subir tu dataset
from google.colab import files
import json
import os

print("📁 Sube tu archivo atomo-llm-dataset.jsonl")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f"✅ Archivo recibido: {filename}")

# Verificar contenido
with open(filename) as f:
    lines = f.readlines()
print(f"📊 Entradas: {len(lines)}")
print(f"📝 Primera entrada:\n{lines[0][:200]}...")

In [ ]:
# 🔄 3. Formatear dataset para entrenamiento (ChatML)

dataset_raw = []
with open(filename) as f:
    for line in f:
        dataset_raw.append(json.loads(line))

def format_chat(entry):
    """Convierte a formato ChatML"""
    system = entry.get("system", "Eres un asistente de Átomo Studio.")
    instruction = entry.get("instruction", "")
    output = entry.get("output", "")
    
    return {
        "conversations": [
            {"from": "system", "value": system},
            {"from": "human", "value": instruction},
            {"from": "gpt", "value": output}
        ]
    }

formatted = [format_chat(e) for e in dataset_raw]

with open("formatted_dataset.json", "w") as f:
    json.dump(formatted, f, ensure_ascii=False)

print(f"✅ Formateadas {len(formatted)} entradas")
print(f"✅ Guardado como formatted_dataset.json")

In [ ]:
# 🏗️ 4. Cargar modelo base (Llama 3.2 3B) con Unsloth
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Contexto máximo
dtype = None  # Auto-detect
load_in_4bit = True  # Cuantización 4-bit para ahorrar VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    device_map="auto",
)

print(f"✅ Modelo cargado: Llama 3.2 3B Instruct")
print(f"   Parámetros: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

In [ ]:
# 🔧 5. Configurar LoRA (fine-tuning eficiente)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Rank de LoRA (más alto = más capacidad de aprendizaje)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("✅ LoRA configurado")
print(f"   Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

In [ ]:
# 🗂️ 6. Tokenizar dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",  # Formato ChatML
)

def apply_template(examples):
    texts = []
    for conv in examples["conversations"]:
        text = tokenizer.apply_chat_template(conv, tokenize=False)
        texts.append(text)
    return {"text": texts}

from datasets import Dataset

hf_dataset = Dataset.from_list(formatted)
dataset = hf_dataset.map(apply_template, batched=True)

print(f"✅ Dataset tokenizado: {len(dataset)} ejemplos")
print(f"\n📄 Ejemplo:\n{dataset[0]['text'][:300]}...")

In [ ]:
# 🎯 7. ENTRENAR
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,  # 3 épocas sobre tu dataset
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",  # Sin wandb/tensorboard
    ),
)

print("🚀 COMENZANDO ENTRENAMIENTO...")
print(f"   Épocas: 3")
print(f"   Batch size: {2 * 4} (efectivo)")
print(f"   Dataset: {len(dataset)} ejemplos")
print(f"")
trainer_stats = trainer.train()
print(f"\n✅ ENTRENAMIENTO COMPLETADO!")

In [ ]:
# 🧪 8. Probar el modelo fine-tuneado
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

prompts = [
    "¿Qué sitios tenemos activos?",
    "¿Cómo levanto el túnel de Cloudflare?",
    "¿Cuál es la visión de Átomo Studio?",
]

for prompt in prompts:
    messages = [
        {"role": "system", "content": "Eres Prometheus 🔥, marketing AI de Átomo Studio."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    print(f"\n❓ {prompt}")
    print("=" * 40)
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extraer solo la respuesta del asistente
    if "assistant" in response:
        response = response.split("assistant\n")[-1] if "\nassistant\n" in response else response.split("assistant")[-1]
    print(f"💬 {response.strip()}\n")

In [ ]:
# 💾 9. Guardar modelo

# Guardar en formato Hugging Face (LoRA adapters)
model.save_pretrained("atomo-llm-lora")
tokenizer.save_pretrained("atomo-llm-lora")
print("✅ LoRA adapters guardados en atomo-llm-lora/")

# Fusionar LoRA con el modelo base y guardar completo
model.save_pretrained_merged("atomo-llm-completo", tokenizer, save_method="merged_16bit")
print("✅ Modelo completo (16bit) guardado en atomo-llm-completo/")

In [ ]:
# 📦 10. Exportar a GGUF para Ollama

# Convertir el modelo completo a formato GGUF (el que usa Ollama)
model.save_pretrained_gguf(
    "atomo-llm-gguf",
    tokenizer,
    quantization_method="q4_k_m",  # Cuantización 4-bit (buena calidad, tamaño pequeño)
)

import os
gguf_files = [f for f in os.listdir("atomo-llm-gguf") if f.endswith(".gguf")]
print(f"✅ Archivos GGUF generados:")
for f in gguf_files:
    size = os.path.getsize(f"atomo-llm-gguf/{f}") / (1024**3)
    print(f"   📄 {f} ({size:.2f} GB)")

In [ ]:
# ⬇️ 11. Descargar el modelo
from google.colab import files
import zipfile
import os

# Comprimir GGUF para descarga
with zipfile.ZipFile("atomo-llm.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk("atomo-llm-gguf"):
        for file in files:
            zf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), "atomo-llm-gguf"))

print("📥 DESCARGANDO...")
print("   El archivo atomo-llm.zip contiene el modelo listo para Ollama")
files.download("atomo-llm.zip")

In [ ]:
# 🏠 12. Instrucciones para importar en Ollama

print('''
═══════════════════════════════════════════════════════
 🏠 CÓMO IMPORTAR EN OLLAMA (en tu PC)
═══════════════════════════════════════════════════════

1. Descarga atomo-llm.zip de Colab
2. Descomprime en tu PC:
   unzip atomo-llm.zip -d ~/atomo-llm

3. Crea un Modelfile:
   echo 'FROM ~/atomo-llm/atomo-llm-q4_k_m.gguf' > Modelfile

4. Importa a Ollama:
   ollama create atomo-llm -f Modelfile

5. Pruébalo:
   ollama run atomo-llm

6. Conéctalo a OpenClaw:
   - Configura el agente para usar ollama/atomo-llm
   - Agent config → model: ollama/atomo-llm
═══════════════════════════════════════════════════════
''')